# Run Qwen3.5 9B model with `mlx-serve`

As you have seen in the last notebooks, the speed of the models depends on the number of 
(active) parameters. In model inference, the speed is dominated by
the availabe memory bandwidth. This is the main reason why GPUs are so much faster in 
text generation compared to CPUs (in addition to prompt parsing).

However, if we can reduce the size of the parameters (not just the number), we could also get
speed increases. On the Mac, this already worked with `mlx`. However, there are more reasons
for running an external LLM server. Notebooks and other programs can access this server
without having to occupy RAM (and the GPU) for each model separately.

On the Mac, you can use `optiq serve` to fire up an Open AI compatible server.

Threfore, run `optiq serve --model mlx-community/gemma-4-12B-it-qat-OptiQ-4bit`

`optiq serve` also offers an Open AI compatible API, it is just running on another port:

In [6]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8080/v1", api_key="sk-optiq-local")

We want to use the notebook for different models:

In [7]:
model = "mlx-community/gemma-4-12B-it-qat-OptiQ-4bit"

## Chat Completion API

Use the chat completion API first:

In [8]:
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "How many 'r's are in 'strawberry'?" } ],
    extra_body={"chat_template_kwargs": {"enable_thinking": True}}
)

print(completion.choices[0].message.content)

There are 3 'r's in 'strawberry'.


The message does not consist of `content` only:

In [9]:
completion.choices[0].message

ChatCompletionMessage(content="There are 3 'r's in 'strawberry'.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='\nThe user is asking for the count of the letter \'r\' in the word "strawberry".\n\n    *   Word: "strawberry"\n    *   Letters: s, t, r, a, w, b, e, r, r, y\n\n    *   s (no)\n    *   t (no)\n    *   r (1)\n    *   a (no)\n    *   w (no)\n    *   b (no)\n    *   e (no)\n    *   r (2)\n    *   r (3)\n    *   y (no)\n\n    *   Total count = 3.')

In [11]:
from IPython.display import display, Markdown

In [12]:
display(Markdown(completion.choices[0].message.content))

There are 3 'r's in 'strawberry'.

In [15]:
# slight change here compared to SGLang: reasoning_content => reasoning
display(Markdown(completion.choices[0].message.reasoning))


The user is asking for the count of the letter 'r' in the word "strawberry".

    *   Word: "strawberry"
    *   Letters: s, t, r, a, w, b, e, r, r, y

    *   s (no)
    *   t (no)
    *   r (1)
    *   a (no)
    *   w (no)
    *   b (no)
    *   e (no)
    *   r (2)
    *   r (3)
    *   y (no)

    *   Total count = 3.

Now switch to the more modern responses API, which here also support the `extra_body`:

In [18]:
response = client.responses.create(model=model, 
                                   input="How many 'r's are in 'strawberry'?",
                                   extra_body={"chat_template_kwargs": {"enable_thinking": True}})

display(Markdown(response.output_text))

There are 3 'r's in 'strawberry'.

Examine the response:

In [19]:
response

Response(id='resp_167b430ea67546f3b45826ef', created_at=1790264516.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='mlx-community/gemma-4-12B-it-qat-OptiQ-4bit', object='response', output=[ResponseReasoningItem(id='rs_9d01a7b0fe9f4760a194bcd5', summary=[Summary(text='\nThe user is asking for the count of the letter \'r\' in the word "strawberry".\n\n    *   Word: "strawberry"\n    *   Letters: s, t, r, a, w, b, e, r, r, y\n\n    *   s (no)\n    *   t (no)\n    *   r (1)\n    *   a (no)\n    *   w (no)\n    *   b (no)\n    *   e (no)\n    *   r (2)\n    *   r (3)\n    *   y (no)\n\n    *   Total count = 3.', type='summary_text')], type='reasoning', content=None, encrypted_content=None, status='completed'), ResponseOutputMessage(id='msg_cb3bcf9458c04e16a4e32f74', content=[ResponseOutputText(annotations=[], text="There are 3 'r's in 'strawberry'.", type='output_text', logprobs=None)], role='assistant', status='completed', type='message', phase=None)], paralle

In [20]:
display(Markdown(response.output[0].content[0].text))

TypeError: 'NoneType' object is not subscriptable

In [ ]:
display(Markdown(response.output[1].content[0].text))

Can we stop the reasoning process for a specific question?

In [ ]:
response = client.responses.create(model=model, 
                                   input="How many 'r's are in 'strawberry'?",
                                   extra_body={ "chat_template_kwargs": {"enable_thinking": False} } )

response.output

Unfortunately, this does not work. We have to go back to the original chat completion API to make it work:

In [ ]:
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "How many 'r's are in 'strawberry'?" } ],
    extra_body={ "chat_template_kwargs": {"enable_thinking": False} }
)

completion